# Qwen2.5-3B-Instruct GPTQ-Int4 — Kaggle T4x2 Serving
## OpenAI-compatible REST API · cloudflared public tunnel · optional API key

| Setting | Value |
|---------|-------|
| Model | `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` |
| Quantization | GPTQ Int4 (~2 GB vs ~6 GB float16) |
| GPUs | 2 x T4 (16 GB each) via `device_map="auto"` |
| Endpoint | OpenAI `POST /v1/chat/completions` |
| Tunnel | cloudflare quick tunnel — no account needed |
| Auth | Optional `Authorization: Bearer <key>` |

> All inference parameters (max_tokens, temperature, top_p, concurrency) are
> centralised in **Cell 2** — change them once, they propagate everywhere.
>
> **Other GPTQ options:** `"Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"`, `"Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4"`.


In [ ]:
!pip install -q optimum
!pip install -q auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu118/
!pip install -q fastapi "uvicorn[standard]" pydantic accelerate


In [ ]:
# ── All tunable parameters — edit here, nowhere else ─────────────────────────

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4"
# Swap to: "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4" or "Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4"

PORT = 8000

# Leave "" for open access; set a string to require Bearer auth.
API_KEY = ""   # e.g. "my-secret-42"

# Inference defaults — callers can override per-request in the JSON body.
MAX_NEW_TOKENS_DEFAULT  = 512
TEMPERATURE_DEFAULT     = 0.7
TOP_P_DEFAULT           = 0.9

# Max simultaneous /v1/chat/completions calls.
# GPU serialises inference anyway; this prevents memory spikes from
# concurrent tokenisation + KV-cache allocation. 4 is a safe default on T4x2.
MAX_CONCURRENT_REQUESTS = 4


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, re, time, subprocess, threading, asyncio, uuid
import torch
from datetime import datetime

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
n_gpu = torch.cuda.device_count()
print(f"GPUs     : {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB")
print(f"model               : {MODEL_NAME}")
print(f"max_tokens default  : {MAX_NEW_TOKENS_DEFAULT}")
print(f"temperature default : {TEMPERATURE_DEFAULT}")
print(f"top_p default       : {TOP_P_DEFAULT}")
print(f"max concurrency     : {MAX_CONCURRENT_REQUESTS}")
print(f"auth                : {'bearer token required' if API_KEY else 'disabled (open access)'}")
print("=" * 60)

if n_gpu == 0:
    raise RuntimeError("No GPU found — enable GPU accelerator in Kaggle settings.")


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import login
    login(token=_tok, add_to_git_credential=False)
    print("HuggingFace: authenticated via Kaggle Secret HF_TOKEN")
except Exception as _e:
    print(f"HuggingFace: no token ({_e}) — continuing unauthenticated")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GPTQConfig

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loading GPTQ-Int4 model (first run downloads ~2 GB) ...")
_t0 = time.time()

# disable_exllama=False uses the fast ExLlama CUDA kernel on T4 (much faster).
_gptq_cfg = GPTQConfig(bits=4, disable_exllama=False)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=_gptq_cfg,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

# model.device is unreliable with device_map="auto" — read hf_device_map instead.
INPUT_DEVICE = next(iter(model.hf_device_map.values()))   # typically "cuda:0"

_elapsed = time.time() - _t0
_params  = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nLoaded {_elapsed:.1f}s  |  {_params:.2f}B params  |  input device: {INPUT_DEVICE}")
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  cuda:{i} VRAM {used:.2f} / {total:.1f} GB")


In [ ]:
from fastapi import FastAPI, HTTPException, Depends, Security
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Qwen Inference API GPTQ-Int4", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

# ── Auth ──────────────────────────────────────────────────────────────────────
_http_bearer = HTTPBearer(auto_error=False)

async def check_auth(creds: HTTPAuthorizationCredentials = Security(_http_bearer)):
    if not API_KEY:
        return
    if creds is None or creds.credentials != API_KEY:
        raise HTTPException(status_code=401, detail="Missing or invalid API key")

# ── Concurrency limiter ───────────────────────────────────────────────────────
# GPU runs one generation at a time anyway; the semaphore queues callers
# instead of letting them pile up or OOM. Set MAX_CONCURRENT_REQUESTS in Cell 2.
_infer_sem = threading.Semaphore(MAX_CONCURRENT_REQUESTS)

# ── Pydantic models ───────────────────────────────────────────────────────────
class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = MODEL_NAME
    messages: list[Message]
    max_tokens: int = MAX_NEW_TOKENS_DEFAULT       # from Cell 2
    temperature: float = TEMPERATURE_DEFAULT        # from Cell 2
    top_p: float = TOP_P_DEFAULT                   # from Cell 2
    do_sample: Optional[bool] = None               # auto-set from temperature

# ── Routes ────────────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "config": {
            "max_new_tokens_default": MAX_NEW_TOKENS_DEFAULT,
            "temperature_default": TEMPERATURE_DEFAULT,
            "top_p_default": TOP_P_DEFAULT,
            "max_concurrent_requests": MAX_CONCURRENT_REQUESTS,
        },
        "auth_required": bool(API_KEY),
        "gpus": [
            {
                "index": i,
                "name": torch.cuda.get_device_properties(i).name,
                "vram_used_gb":  round(torch.cuda.memory_allocated(i)  / 1e9, 2),
                "vram_total_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1),
            }
            for i in range(torch.cuda.device_count())
        ],
        "ts": datetime.utcnow().isoformat() + "Z",
    }

@app.get("/v1/models", dependencies=[Depends(check_auth)])
def list_models():
    return {
        "object": "list",
        "data": [{"id": MODEL_NAME, "object": "model", "owned_by": "local"}],
    }

@app.post("/v1/chat/completions", dependencies=[Depends(check_auth)])
def chat(req: ChatRequest):
    # Block until a slot is free (MAX_CONCURRENT_REQUESTS governs this)
    acquired = _infer_sem.acquire(timeout=25)
    if not acquired:
        raise HTTPException(status_code=503,
            detail=f"Server busy — {MAX_CONCURRENT_REQUESTS} slots full; retry later")
    try:
        # Build prompt from chat template
        prompt = tokenizer.apply_chat_template(
            [{"role": m.role, "content": m.content} for m in req.messages],
            tokenize=False,
            add_generation_prompt=True,
        )

        # Tokenise — send to INPUT_DEVICE (first model layer's device)
        enc = tokenizer(prompt, return_tensors="pt").to(INPUT_DEVICE)
        prompt_len = enc["input_ids"].shape[1]

        temp      = float(req.temperature)
        do_sample = req.do_sample if req.do_sample is not None else (temp > 0)
        gen_kw = dict(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            max_new_tokens=req.max_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if do_sample:
            gen_kw["temperature"] = temp
            gen_kw["top_p"] = float(req.top_p)

        t0 = time.perf_counter()
        with torch.no_grad():
            out_ids = model.generate(**gen_kw)
        latency_ms = (time.perf_counter() - t0) * 1000

        new_ids = out_ids[0][prompt_len:]
        answer  = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
        n_out   = len(new_ids)

        return {
            "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": req.model,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": answer},
                "finish_reason": "stop",
            }],
            "usage": {
                "prompt_tokens":     prompt_len,
                "completion_tokens": n_out,
                "total_tokens":      prompt_len + n_out,
                "latency_ms":        round(latency_ms, 1),
                "tokens_per_sec":    round(n_out / (latency_ms / 1000), 1),
            },
        }
    finally:
        _infer_sem.release()

print("FastAPI app ready.")
print("  GET  /health")
print("  GET  /v1/models")
print("  POST /v1/chat/completions")


In [ ]:
import uvicorn

# Stop any server left over from a previous run of this cell.
try:
    _server.should_exit = True
    time.sleep(1.5)
except NameError:
    pass

# Run uvicorn in a DEDICATED thread with its OWN event loop.
# Sharing the Jupyter kernel's loop (via nest_asyncio) causes context
# conflicts when aiohttp also runs async IO on that same loop.
_server = None

def _run_server():
    global _server
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    cfg = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
    _server = uvicorn.Server(cfg)
    loop.run_until_complete(_server.serve())

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(3)
print(f"Server: http://localhost:{PORT}")

CF_BIN = "/tmp/cloudflared"
if not os.path.exists(CF_BIN):
    print("Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q", "-O", CF_BIN,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    subprocess.run(["chmod", "+x", CF_BIN], check=True)

cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)

public_url = None
deadline = time.time() + 45
while time.time() < deadline:
    line = cf_proc.stderr.readline()
    if not line:
        time.sleep(0.2)
        continue
    hit = re.search(r"https://[a-zA-Z0-9-]+[.]trycloudflare[.]com", line)
    if hit:
        public_url = hit.group(0)
        break

if not public_url:
    try:
        remaining = cf_proc.stderr.read(2000)
        hit = re.search(r"https://[a-zA-Z0-9-]+[.]trycloudflare[.]com", remaining)
        if hit:
            public_url = hit.group(0)
    except Exception:
        pass

if public_url:
    print()
    print("=" * 60)
    print("PUBLIC URL:")
    print(f"  {public_url}")
    print(f"\n  GET  {public_url}/health")
    print(f"  GET  {public_url}/v1/models")
    print(f"  POST {public_url}/v1/chat/completions")
    if API_KEY:
        print(f"\n  Authorization: Bearer {API_KEY}")
    else:
        print("\n  No auth required")
    print("=" * 60)
else:
    print("WARNING: cloudflare URL not found — try: cf_proc.stderr.read(2000)")


In [ ]:
import requests

url  = f"http://localhost:{PORT}/v1/chat/completions"
hdrs = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

payload = {
    "model": "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "What is 2 + 2? Answer in one word."},
    ],
    "max_tokens": 20,
    "temperature": 0.0,
}

resp = requests.post(url, json=payload, headers=hdrs, timeout=60)
resp.raise_for_status()
data = resp.json()

print("Answer     :", data["choices"][0]["message"]["content"])
print("Usage      :", data["usage"])


In [ ]:
# ==============================================================================
# ⚙️  STRESS TEST CONFIGURATION
# ==============================================================================

API_TYPE = "openai"   # "openai" | "google"

# Paste the public URL printed by Cell 7 (cloudflared / ngrok)
API_URL  = "https://YOUR-TUNNEL.trycloudflare.com/v1/chat/completions"
API_KEY  = ""          # match API_KEY in Cell 2, or leave "" for open access

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4"   # must match what the server is serving

TOTAL_REQUESTS    = 1000
CONCURRENCY_LIMIT = 4     # keep <= MAX_CONCURRENT_REQUESTS on the server
                          # cloudflared kills connections after 30s; with 32
                          # concurrent requests queueing, most wait >30s -> 524
# CONCURRENCY_LIMIT = 32  # use this only if the server has many workers (e.g. vLLM)
MAX_TOKENS        = 128
MAX_RETRIES       = 3     # exponential-backoff retries on 429/5xx

PROMPTS = [
    "Explain quantum entanglement to a 5-year-old.",
    "Write a haiku about a server crashing.",
    "List 5 fun facts about dolphins.",
    "Translate 'Hello world' into Python code.",
    "What is the capital of Australia?",
    "Summarise the plot of Romeo and Juliet in one sentence.",
    "Why is the sky blue?",
    "Write a short email declining a wedding invitation politely.",
    "Explain the difference between TCP and UDP.",
    "Give me a recipe for pancakes.",
]

# ==============================================================================
# 🚀  ENGINE
# ==============================================================================

import asyncio, aiohttp, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class StressTester:
    def __init__(self):
        self.results   = []
        self.start_time = 0
        self.end_time   = 0

    # ── Payload builder (supports multiple API types) ─────────────────────────
    def get_payload(self, prompt):
        if API_TYPE == "openai":
            return {
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": MAX_TOKENS,
                "temperature": 0.7,
            }
        elif API_TYPE == "google":
            return {
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {"maxOutputTokens": MAX_TOKENS},
            }

    def get_token_count(self, data):
        try:
            if API_TYPE == "openai":
                return data["usage"]["completion_tokens"]
            elif API_TYPE == "google":
                return data.get("usageMetadata", {}).get("candidatesTokenCount", 0)
        except Exception:
            return 0

    # ── Single worker with exponential-backoff retry ──────────────────────────
    async def worker(self, session, semaphore, req_id):
        prompt = random.choice(PROMPTS)
        t0 = time.time()

        headers = {"Content-Type": "application/json", "User-Agent": "stress-test/2.0"}
        if API_KEY:
            headers["Authorization"] = f"Bearer {API_KEY}"

        async with semaphore:
            last_err = None
            for attempt in range(MAX_RETRIES + 1):
                try:
                    async with session.post(
                        API_URL, json=self.get_payload(prompt), headers=headers
                    ) as resp:
                        latency = time.time() - t0
                        body    = await resp.text()

                        if resp.status == 200:
                            data   = json.loads(body)
                            tokens = self.get_token_count(data)
                            self.results.append({
                                "id": req_id, "status": "success",
                                "latency": latency, "tokens": tokens,
                                "timestamp": time.time(),
                            })
                            return

                        # Retry on transient errors
                        if resp.status in (429, 500, 502, 503, 504) and attempt < MAX_RETRIES:
                            await asyncio.sleep(2 ** attempt + random.uniform(0, 0.5))
                            last_err = f"{resp.status}: {body[:200]}"
                            continue

                        self.results.append({
                            "id": req_id, "status": "error",
                            "latency": latency, "tokens": 0,
                            "error_msg": f"{resp.status}: {body[:200]}",
                            "timestamp": time.time(),
                        })
                        return

                except Exception as exc:
                    last_err = str(exc)
                    if attempt < MAX_RETRIES:
                        await asyncio.sleep(2 ** attempt + random.uniform(0, 0.5))
                    else:
                        self.results.append({
                            "id": req_id, "status": "exception",
                            "latency": time.time() - t0, "tokens": 0,
                            "error_msg": last_err, "timestamp": time.time(),
                        })

    # ── Main run loop ─────────────────────────────────────────────────────────
    async def run(self):
        print(f"🔥 STARTING STRESS TEST")
        print(f"   Requests    : {TOTAL_REQUESTS}")
        print(f"   Concurrency : {CONCURRENCY_LIMIT}")
        print(f"   Max-tokens  : {MAX_TOKENS}")
        print(f"   Retries     : {MAX_RETRIES}")
        print(f"   Target      : {API_URL}")
        print(f"   Model       : {MODEL_NAME}")
        print()

        self.start_time = time.time()
        sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
        timeout   = aiohttp.ClientTimeout(total=180)
        connector = aiohttp.TCPConnector(
            limit=CONCURRENCY_LIMIT, limit_per_host=CONCURRENCY_LIMIT
        )

        async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
            tasks = [self.worker(session, sem, i) for i in range(TOTAL_REQUESTS)]
            done  = 0
            step  = max(1, TOTAL_REQUESTS // 10)
            for fut in asyncio.as_completed(tasks):
                await fut
                done += 1
                if done % step == 0:
                    pct = done / TOTAL_REQUESTS * 100
                    ok  = sum(1 for r in self.results if r["status"] == "success")
                    print(f"   Progress: {done:>4}/{TOTAL_REQUESTS}  ({pct:3.0f}%)  ✅ {ok}")

        self.end_time = time.time()
        print("\n✅ TEST COMPLETE.")

    # ── Analysis + reporting ──────────────────────────────────────────────────
    def analyze(self):
        df       = pd.DataFrame(self.results)
        ok       = df[df["status"] == "success"].copy()
        duration = self.end_time - self.start_time

        total_tokens = int(ok["tokens"].sum()) if not ok.empty else 0
        rps          = len(ok) / duration if duration else 0
        tps          = total_tokens / duration if duration else 0
        avg_lat      = ok["latency"].mean() if not ok.empty else 0
        fail_rate    = (len(df) - len(ok)) / len(df) * 100 if len(df) else 0

        lats = ok["latency"].values if not ok.empty else np.array([0])
        p50  = np.percentile(lats, 50)
        p95  = np.percentile(lats, 95)
        p99  = np.percentile(lats, 99)

        print("\n" + "=" * 45)
        print("📊  PERFORMANCE METRICS")
        print("=" * 45)
        print(f"⏱️   Duration          : {duration:.2f} s")
        print(f"📨  Total requests    : {len(df)}")
        print(f"✅  Successful        : {len(ok)}")
        print(f"❌  Failed            : {len(df) - len(ok)}  ({fail_rate:.2f}%)")
        print(f"🚀  Throughput (RPS)  : {rps:.2f} req/s")
        print(f"⚡  Token gen speed   : {tps:.2f} tok/s")
        print(f"⏳  Avg latency       : {avg_lat:.3f} s")
        print(f"📈  P50 latency       : {p50:.3f} s")
        print(f"📈  P95 latency       : {p95:.3f} s")
        print(f"📈  P99 latency       : {p99:.3f} s")
        print("=" * 45)

        errs = df[df["status"] != "success"]
        if not errs.empty:
            print("\n🚨  FIRST ERROR SAMPLE:")
            print(errs.iloc[0].get("error_msg", "unknown"))

        if not ok.empty:
            self._plot(ok, df)

    def _plot(self, ok, full_df):
        plt.style.use("bmh")
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

        # Latency histogram with percentile lines
        ax1.hist(ok["latency"], bins=30, color="#3498db", edgecolor="white")
        for pct, col, lbl in [
            (50, "#2ecc71", "P50"), (95, "#f39c12", "P95"), (99, "#e74c3c", "P99")
        ]:
            v = np.percentile(ok["latency"], pct)
            ax1.axvline(v, color=col, linestyle="--", linewidth=1.5, label=f"{lbl} {v:.2f}s")
        ax1.set_title("Latency Distribution")
        ax1.set_xlabel("Seconds")
        ax1.set_ylabel("Count")
        ax1.legend(fontsize=8)

        # Requests-per-second timeline
        ok = ok.copy()
        ok["datetime"] = pd.to_datetime(ok["timestamp"], unit="s")
        ts = ok.set_index("datetime").resample("1s").count()["id"]
        ax2.plot(ts.index, ts.values, color="#2ecc71", linewidth=2)
        ax2.fill_between(ts.index, ts.values, alpha=0.2, color="#2ecc71")
        ax2.set_title("Successful Requests / Second")
        ax2.set_xlabel("Time")
        ax2.set_ylabel("req/s")

        # Success / failure pie chart
        counts = full_df["status"].value_counts()
        cmap   = {"success": "#2ecc71", "error": "#e74c3c", "exception": "#f1c40f"}
        colors = [cmap.get(s, "#95a5a6") for s in counts.index]
        ax3.pie(
            counts, labels=counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90, wedgeprops=dict(edgecolor="white"),
        )
        ax3.set_title("Success vs Failure Rate")

        plt.tight_layout()
        plt.show()

# ==============================================================================
# ▶  RUN
# ==============================================================================

tester = StressTester()
await tester.run()
tester.analyze()


## Usage from outside Kaggle

Replace `PUBLIC_URL` with the URL printed by Cell 7.

### curl — no auth
```bash
curl -s -X POST PUBLIC_URL/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -d '{
        "model": "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
        "messages": [{"role":"user","content":"Hello!"}],
        "max_tokens": 200,
        "temperature": 0.7
      }' | python -m json.tool
```

### curl — with API key
```bash
curl -s -X POST PUBLIC_URL/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer my-secret-42" \\
  -d '{"model":"Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4","messages":[{"role":"user","content":"Hi"}],"max_tokens":100}' \\
  | python -m json.tool
```

### Python openai SDK
```python
from openai import OpenAI

client = OpenAI(base_url="PUBLIC_URL/v1", api_key="my-secret-42")

resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "Explain LoRA in two sentences."},
    ],
    max_tokens=300,
    temperature=0.7,
)
print(resp.choices[0].message.content)
```

### Stop the tunnel + server
```python
cf_proc.terminate()
_server.should_exit = True
```
